# 03 · 캐시와 버전 경계

적중률보다 같은 답을 재사용해도 되는 조건부터 정의합니다. 권장 15분.

위에서 아래로 실행합니다. 각 실험은 새 앱을 사용하며 실제 LLM·GPU·외부 API는 기본 경로에 없습니다. 먼저 결과를 예상하고 실행한 뒤 변수 하나를 바꾸고 관찰을 기록하세요.

In [ ]:
from pathlib import Path
import sys, asyncio, json, time
from dataclasses import replace
import numpy as np
import httpx
from IPython.display import display
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from week09.lab.app import create_app
from week09.lab.service import Settings, ChatRequest, Service, TTLCache, TokenBucket, cache_key
from week09.lab.benchmark import run_load, summarize
assert sys.version_info[:2] == (3,11), 'Python 3.11 .venv 커널 선택'
OUT = ROOT/'output/validation/week09-lab'
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0],'NumPy',np.__version__,'HTTPX',httpx.__version__)


## 같은 요청 두 번

두 번째 요청의 캐시 적중과 제공자 호출 수를 확인합니다.

In [ ]:
app = create_app()
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    a = (await client.post('/api/v1/chat',json={'message':'반품 정책'})).json()
    b = (await client.post('/api/v1/chat',json={'message':'반품 정책'})).json()
assert not a['cached'] and b['cached'] and app.state.service.provider_calls == 1
display([a,b])

## 파라미터와 버전

모델·제공자·프롬프트·출력 스키마·생성 설정은 캐시 의미를 바꿉니다.

In [ ]:
req = ChatRequest(message='x')
cfg = Settings()
keys = [cache_key(req,cfg), cache_key(req,replace(cfg,prompt_version='v2')),cache_key(req.model_copy(update={'max_tokens':32}),cfg)]
assert len(set(keys)) == 3
display(keys)

## 권한 경계

아래는 설계 명세입니다. 이번 앱에는 인증·tenant·RAG 문서가 없어 다중 사용자 권한 분리를 구현하지 않습니다.

In [ ]:
production_key_spec = ['인증된 tenant/권한','질문','모델 revision','프롬프트 버전','문서 revision','생성 설정','출력 schema']
assert '인증된 tenant/권한' in production_key_spec
display(production_key_spec)

## 만료 경계

가짜 시계로 빠르게 실험합니다. 만료 시각과 같으면 적중하지 않습니다.

In [ ]:
clock = [0.]
cache = TTLCache(2,2,clock=lambda:clock[0])
cache.set('A','version1')
assert cache.get('A') == 'version1'
clock[0] = 2
assert cache.get('A') is None

## 메모리 상한

가장 최근에 접근하지 않은 항목부터 제거합니다. 용량 상한이 없으면 긴 수업에서도 메모리가 누적됩니다.

In [ ]:
cache = TTLCache(30,2)
cache.set('A','a'); cache.set('B','b'); cache.get('A'); cache.set('C','c')
assert cache.get('B') is None and len(cache.values) == 2
print(list(cache.values))

## 동시에 같은 질문

캐시는 진행 중인 작업을 합치지 않습니다. 콜드 캐시에서 중복 호출이 발생하는 반례입니다.

In [ ]:
app = create_app(Settings(max_inflight=2))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    responses = await asyncio.gather(*(client.post('/api/v1/chat',json={'message':'동일 질문'}) for _ in range(2)))
assert app.state.service.provider_calls == 2
assert all(not r.json()['cached'] for r in responses)
print('중복 진행 작업 병합(single-flight)은 별도 확장 과제')

## 실패 응답의 재사용 방지

장애 주입은 성공 캐시를 우회합니다. 오류를 정상 답변으로 캐시하지 않습니다.

In [ ]:
app = create_app(Settings(timeout_s=.01))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    failed = await client.post('/api/v1/chat',json={'message':'x','scenario':'timeout'})
assert failed.status_code == 504 and len(app.state.service.cache.values) == 0

## 캐시 비활성화

같은 질문으로도 제공자를 매번 호출할 수 있습니다. 부하 비교에서는 조건을 고정합니다.

In [ ]:
app = create_app()
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    for _ in range(2):
        await client.post('/api/v1/chat',json={'message':'x','use_cache':False})
assert app.state.service.provider_calls == 2

## 캐시 삭제와 개인정보

해시는 암호화가 아니며 짧은 입력은 추측될 수 있습니다. 원문·캐시값·백업·로그를 함께 다룹니다.

In [ ]:
policy = {'allow':'합성 공개 예시만','retention':'30초, 최대128항목','restart':'메모리 초기화','production_gap':'권한별 분리·암호화·삭제 감사·다중 worker 동기화'}
(OUT/'cache-policy.json').write_text(json.dumps(policy,ensure_ascii=False,indent=2),encoding='utf-8')
display(policy)

## 포트폴리오 기록

변경 변수 / 고정 조건 / 실제 관측값 / 실패 사례 / 해석의 한계 / 다음 확인을 기록하세요. 개인정보·학생 원문은 공개 저장소에 넣지 않습니다. 별도 과제 마감은 소속 공지를 따르며 이 실습에서 새 제출 기한을 만들지 않습니다.